# Hugging Face Dataset Ingestion to Normalized CSV

This notebook loads a Hugging Face dataset and exports a raw CSV compatible with the manifest pipeline.

## 2) Configure source and output

In [ ]:
from pathlib import Path

HF_DATASET_ID = "nyuuzyou/suno"
HF_SPLIT = "train"

# Normalize source naming for your manifest
SOURCE_NAME = "hf_nyuuzyou/suno"
GENERATOR = "suno"


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find project root from current working directory")


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_CSV = PROJECT_ROOT / "data/raw/hf_suno.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset:      {HF_DATASET_ID} [{HF_SPLIT}]")
print(f"Output:       {OUTPUT_CSV}")


Dataset: nyuuzyou/suno [train]
Output:  data\raw\hf_suno.csv


## 3) Load dataset and inspect columns

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(HF_DATASET_ID, split=HF_SPLIT)
df = ds.to_pandas()

print(f"Rows: {len(df)}")
print("Columns:")
print(sorted(df.columns.tolist()))
df.head(3)

c:\Users\aurel\Projects\deezer-robustness-project\robust-deepfake-detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aurel\Projects\deezer-robustness-project\robust-deepfake-detector\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aurel\.cache\huggingface\hub\datasets--nyuuzyou--suno. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to act

Rows: 659788
Columns:
['audio_url', 'avatar_image_url', 'created_at', 'display_name', 'handle', 'id', 'image_large_url', 'image_url', 'is_handle_updated', 'is_liked', 'is_public', 'is_trashed', 'is_video_pending', 'major_model_version', 'metadata_artist_clip_id', 'metadata_concat_history', 'metadata_cover_clip_id', 'metadata_duration', 'metadata_edit_session_id', 'metadata_error_message', 'metadata_error_type', 'metadata_gpt_description_prompt', 'metadata_has_vocal', 'metadata_history', 'metadata_infill', 'metadata_is_audio_upload_tos_accepted', 'metadata_negative_tags', 'metadata_persona_id', 'metadata_prompt', 'metadata_refund_credits', 'metadata_stem_from_id', 'metadata_stream', 'metadata_tags', 'metadata_task', 'metadata_type', 'model_name', 'persona', 'play_count', 'status', 'title', 'upvote_count', 'user_id', 'video_url']


,id,video_url,audio_url,image_url,image_large_url,is_video_pending,major_model_version,model_name,is_liked,user_id,...,metadata_artist_clip_id,metadata_cover_clip_id,metadata_edit_session_id,metadata_stem_from_id,metadata_persona_id,metadata_task,metadata_is_audio_upload_tos_accepted,metadata_concat_history,metadata_history,metadata_infill
0,00004802-8ba3-43a0-b813-2b46c636ca7c,https://cdn1.suno.ai/00004802-8ba3-43a0-b813-2...,https://cdn1.suno.ai/00004802-8ba3-43a0-b813-2...,https://cdn2.suno.ai/image_f5262127-4b61-420a-...,https://cdn2.suno.ai/image_large_f5262127-4b61...,False,v3.5,chirp-v3,False,5f389b7f-7cf0-432f-a0db-dcdbd000155d,...,None,None,None,None,None,None,None,"[{""id"": ""3443d52d-4b49-456a-84d3-22b60269dd73""...",None,None
1,000099e1-5e6c-4ff5-9bc5-e5ef9eee9f29,https://cdn1.suno.ai/000099e1-5e6c-4ff5-9bc5-e...,https://cdn1.suno.ai/000099e1-5e6c-4ff5-9bc5-e...,https://cdn2.suno.ai/image_000099e1-5e6c-4ff5-...,https://cdn2.suno.ai/image_large_000099e1-5e6c...,False,v3.5,chirp-v3,False,17fdafa3-579c-4201-bc3d-019d3f5c193d,...,None,None,None,None,None,None,None,None,None,None
2,0000ea1f-027c-417d-9b45-42700040f10a,https://cdn1.suno.ai/0000ea1f-027c-417d-9b45-4...,https://cdn1.suno.ai/0000ea1f-027c-417d-9b45-4...,https://cdn2.suno.ai/image_0000ea1f-027c-417d-...,https://cdn2.suno.ai/image_large_0000ea1f-027c...,False,v3.5,chirp-v3,False,5dff1d06-3816-4ae0-b7da-ff6583e591ed,...,None,None,None,None,None,None,None,None,None,None


## 4) Map to normalized raw schema (CSV)

Required by your collection pipeline: `source`, `source_track_id`, `audio_uri`.

Recommended extras: `title`, `artist`, `username`, `generator`, `generator_version`, `metadata`, `label`.

In [4]:
import json

def safe_get(frame: pd.DataFrame, col: str):
    return frame[col] if col in frame.columns else None

normalized = pd.DataFrame({
    "source": SOURCE_NAME,
    "source_track_id": safe_get(df, "id"),
    "audio_uri": safe_get(df, "audio_url"),
    "title": safe_get(df, "title"),
    "artist": safe_get(df, "display_name"),
    "username": safe_get(df, "handle"),
    "generator": GENERATOR,
    "generator_version": safe_get(df, "major_model_version"),
    "label": "ai",
})

metadata_cols = [
    "model_name",
    "metadata_prompt",
    "metadata_tags",
    "metadata_duration",
]

def build_metadata(row):
    payload = {}
    for c in metadata_cols:
        if c in row.index and pd.notna(row[c]):
            payload[c] = row[c]
    return json.dumps(payload, default=str, ensure_ascii=False) if payload else None

normalized["metadata"] = df.apply(build_metadata, axis=1)

normalized = normalized.dropna(subset=["audio_uri"]).copy()
normalized["audio_uri"] = normalized["audio_uri"].astype(str).str.strip()
normalized = normalized[normalized["audio_uri"] != ""].copy()

print(f"Normalized rows: {len(normalized)}")
normalized.head(3)

Normalized rows: 658735


,source,source_track_id,audio_uri,title,artist,username,generator,generator_version,label,metadata
0,hf_suno,00004802-8ba3-43a0-b813-2b46c636ca7c,https://cdn1.suno.ai/00004802-8ba3-43a0-b813-2...,THE INNER WORLD 2,ThreeSeven,threeseven3790,suno,v3.5,ai,"{""model_name"": ""chirp-v3"", ""metadata_prompt"": ..."
1,hf_suno,000099e1-5e6c-4ff5-9bc5-e5ef9eee9f29,https://cdn1.suno.ai/000099e1-5e6c-4ff5-9bc5-e...,Rainy Day Sanctuary,BansheeQueenForever2002,royalbanshee2002,suno,v3.5,ai,"{""model_name"": ""chirp-v3"", ""metadata_prompt"": ..."
2,hf_suno,0000ea1f-027c-417d-9b45-42700040f10a,https://cdn1.suno.ai/0000ea1f-027c-417d-9b45-4...,What Would My World Be Without You (Eurobeat S...,GHITALO MUSIC,ghitalomusic,suno,v3.5,ai,"{""model_name"": ""chirp-v3"", ""metadata_prompt"": ..."


## 5) Save CSV for aggregation

In [5]:
normalized.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV} ({len(normalized)} rows)")

Saved: data\raw\hf_suno.csv (658735 rows)
